In [1]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Dynamic Allocation")
    .master("spark://5892aedce22e:7077")
    .config("spark.executor.cores", 2)
    .config("spark.executor.memory", "512M")
    .config("spark.dynamicAllocation.enabled", True)
    .config("spark.dynamicAllocation.minExecutors", 0)
    .config("spark.dynamicAllocation.maxExecutors", 5)
    .config("spark.dynamicAllocation.initialExecutors", 1)
    .config("spark.dynamicAllocation.shuffleTracking.enabled", True)
    .config("spark.dynamicAllocation.executorIdleTimeout", "60s")
    .config("spark.dynamicAllocation.cachedExecutorIdleTimeout", "60s")
    .getOrCreate()
)

spark

In [ ]:
# Enable dynamic allocation (auto scale executors based on workload)
.config("spark.dynamicAllocation.enabled", True)

# Minimum number of executors Spark can scale down to (0 = can fully release resources)
.config("spark.dynamicAllocation.minExecutors", 0)

# Maximum number of executors Spark can scale up to
.config("spark.dynamicAllocation.maxExecutors", 5)

# Number of executors to start the application with
.config("spark.dynamicAllocation.initialExecutors", 1)

# Enable shuffle tracking so Spark can safely remove executors without external shuffle service
.config("spark.dynamicAllocation.shuffleTracking.enabled", True)

# Time after which an idle executor (no tasks running) will be removed
.config("spark.dynamicAllocation.executorIdleTimeout", "60s")

# Time after which an idle executor with cached data will be removed
.config("spark.dynamicAllocation.cachedExecutorIdleTimeout", "60s")

In [2]:
# Read Sales data

sales_schema = "transacted_at string, trx_id string, retailer_id string, description string, amount double, city_id string"

sales = spark.read.format("csv").schema(sales_schema).option("header", True).load("/data/input/new_sales.csv")

In [3]:
# Read City data

city_schema = "city_id string, city string, state string, state_abv string, country string"

city = spark.read.format("csv").schema(city_schema).option("header", True).load("/data/input/cities.csv")

In [4]:
# Join Data

df_sales_joined = sales.join(city, on=sales.city_id==city.city_id, how="left_outer")

In [5]:
df_sales_joined.write.format("noop").mode("overwrite").save()

In [ ]:
# ================================
# Spark Dynamic Allocation - Internal Working Notes
# ================================

# 1. Core Idea:
# Spark dynamically scales executors based on workload.
# If there are pending tasks → scale up
# If executors are idle → scale down

# 2. Scheduler Backend:
# - Tracks pending tasks in the queue
# - Detects backlog (task demand > available cores)
# - Triggers request for more executors

# 3. Executor Allocation Manager:
# - Decides number of executors to add/remove
# - Ensures limits are respected:
#   -> minExecutors
#   -> maxExecutors

# 4. Executor Monitor:
# - Tracks executor state (busy / idle)
# - Measures idle time
# - Checks if executor holds cached data or shuffle data

# 5. Scale-Up Flow:
# - Tasks arrive → queue builds up
# - Spark detects insufficient capacity
# - Requests cluster manager for more executors
# - New executors are launched and assigned tasks

# 6. Scale-Down Flow:
# - Executors become idle after task completion
# - Idle timer starts (executorIdleTimeout)
# - If safe, executor is removed
# - Must respect minExecutors limit

# 7. Shuffle Tracking (IMPORTANT):
# spark.dynamicAllocation.shuffleTracking.enabled = True
# - Tracks shuffle blocks internally
# - Allows safe removal of executors without external shuffle service
# - Prevents shuffle data loss issues

# 8. Idle Timeout Logic:
# - executorIdleTimeout:
#     Removes idle executors after threshold (e.g., 60s)
#
# - cachedExecutorIdleTimeout:
#     Removes executors even if they hold cached data

# 9. Decision Logic (Simplified):
# IF pending_tasks > available_capacity:
#     request_more_executors()
# ELSE IF executor_idle_time > timeout:
#     remove_executor()

# 10. Cluster Manager Role:
# - Spark only requests resources
# - Actual executor creation handled by:
#   -> YARN / Kubernetes / Standalone cluster manager

# 11. Key Internal Benefit:
# - Better resource utilization
# - Lower cost in cloud environments
# - Automatically adapts to workload changes